# Player framing

Associated chart: How Fox frames each player

Output files:
`politicization_people.csv`, `appearance_talk.csv`, `celebrity_talk.csv`, and `agency.csv`
in `src/data/`.

Input files: `data/fox_wnba.csv`, `src/data/headline_network.json`, `data/politicization_terms.csv`, `data/framing_terms.csv`.

Requirements: spaCy

`pip install spacy`
`python -m spacy download en_core_web_sm`


## Setup

In [ ]:
import csv, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUT = Path("../src/data")

fox = pd.read_csv(DATA / "fox_wnba.csv")
headlines = (fox.assign(headline=fox.headline.str.strip())
                .dropna(subset=["headline"])
                .drop_duplicates("headline"))
headlines = [{"t": r.headline, "d": str(r.date)[:10]} for r in headlines.itertuples()]

# every person the piece tracks, with their role (players / politics / media / league)
roster = json.loads((OUT / "people.json").read_text())
groups = {name: p["group"] for name, p in roster.items()}
framing_parts = []
# each person's own pattern from people.json (full names where a surname is
# shared or a word: "cameron brink", never "the brink of suspension")
name_rx = {name: re.compile(p["rx"], re.I) for name, p in roster.items()}

totals = Counter()
for row in headlines:
    for name, rx in name_rx.items():
        if rx.search(row["t"]):
            totals[name] += 1

MIN_HEADLINES = 9
print(f"{len(headlines):,} unique headlines · {len(groups)} people tracked")

## Politicization

Which headlines carry political language, from the term groups in
`data/politicization_terms.csv`.


In [ ]:
# per-headline political groups + per-person machine verdicts
TERM_GROUPS = {}
for _r in csv.DictReader(open(DATA / "politicization_terms.csv")):
    TERM_GROUPS.setdefault(_r["group"], []).append(_r["term"])
TERM_LISTS = TERM_GROUPS
import sys as _sys; _sys.path.insert(0, "."); from framing import guard_term   # "trans" must not match "transition"
TERM_GROUPS = {g: "|".join(rf"\b(?:{guard_term(t)})" for t in ts) for g, ts in TERM_LISTS.items()}
group_rx = {name: re.compile(p, re.I) for name, p in TERM_GROUPS.items()}
term_rx = {g: [re.compile(r"\b(?:" + guard_term(t) + r")\w*", re.I) for t in ts]
           for g, ts in TERM_LISTS.items()}
culture_war_rx = [re.compile(t, re.I) for t in TERM_LISTS["culture_war"]]
def own_culture_war(person):
    return [rx for rx in culture_war_rx if rx.search(person)]

hl_political = {}          # headline -> "; ".join(groups) when politically flagged
hl_fired = {}              # headline -> [(group, matched word)]
pol_ok = set()             # (headline, person) pairs the machine counts as political
for row in headlines:
    matched = [g for g, rx in group_rx.items() if rx.search(row["t"])]
    fired = [(g, m.group(0).lower()) for g in matched
             for rx_t in term_rx[g] for m in rx_t.finditer(row["t"])]
    if any(g != "partisan" for g in matched):
        hl_political[row["t"]] = "; ".join(matched)
        hl_fired[row["t"]] = fired
    for person, rx in name_rx.items():
        if not rx.search(row["t"]):
            continue
        mine = list(matched)
        if "culture_war" in mine:
            own = own_culture_war(person)
            others = [r for r in culture_war_rx if r not in own]
            if not any(r.search(row["t"]) for r in others):
                mine.remove("culture_war")
        if any(g != "partisan" for g in mine):
            pol_ok.add((row["t"], person))
print(f"{len(hl_political)} headlines carry political language")

## Appearance & celebrity treatment

How much of a player's coverage is about her appearance or her private life (celebrity). 


In [ ]:
# vocabularies: data/framing_terms.csv · grammar: notebooks/framing.py
import spacy
import framing

nlp = framing.load_parser(framing.HEADLINE_PARSER, disable=("ner", "lemmatizer"))
appearance_hits = framing.attribute(nlp, headlines, groups, "appearance", name_rx)
celebrity_hits = framing.attribute(nlp, headlines, groups, "celebrity", name_rx)
print(f"{sum(len(v) for v in appearance_hits.values())} appearance attributions · "
      f"{sum(len(v) for v in celebrity_hits.values())} celebrity")

## Agency & victim framing

Who acts and who is acted upon. "Alyssa Thomas punches Caitlin Clark" and "Caitlin Clark
punched in the throat" carry the same tone but opposite framing. 

Agency is how often she's the
subject of a verb versus its object (+1 always acting, −1 always acted upon), from the parse.
Victim counts any headline where language acts on her.


In [ ]:
SUBJ = {"nsubj", "csubj", "agent"}
PASSIVE = {"nsubjpass"}
OBJ = {"dobj", "obj", "iobj", "dative", "oprd"}
CONTACT_VERBS = framing.term_set("agency", "contact_verb")
CONTACT_NOUNS = framing.term_set("agency", "contact_noun")
# reporting verbs say nothing about agency
REPORTING = framing.term_set("agency", "reporting_verb")
MIN_PLACED = 12

nlp_full = framing.load_parser(framing.HEADLINE_PARSER)
players = {name for name, g in groups.items() if g == "players"}
surname = {name: name.split()[-1] for name in players}

placed = defaultdict(lambda: {"agent": 0, "patient": 0, "contact_by": 0, "mentions": 0,
                              "as_agent": Counter(), "as_patient": Counter()})
texts = [r["t"] for r in headlines]
for headline, doc in zip(texts, nlp_full.pipe(texts, batch_size=64)):
    for name, last in surname.items():
        if not re.search(r"\b" + re.escape(last) + r"\b", headline):
            continue
        stats = placed[name]
        stats["mentions"] += 1
        token = next((t for t in doc if re.sub(r"['’]s$", "", t.text) == last), None)
        if token is None:
            continue
        node, verb = token, None
        for _ in range(4):
            head = node.head
            if head is node:
                break
            if head.pos_ in {"VERB", "AUX"}:
                verb = head
                break
            node = head
        if verb is None or verb.lemma_.lower() in REPORTING:
            continue
        lemma = verb.lemma_.lower()
        physical = lemma in CONTACT_VERBS or any(
            c.text.lower() in CONTACT_NOUNS for c in verb.children)
        if token.dep_ in SUBJ:
            stats["agent"] += 1
            stats["as_agent"][lemma] += 1
            if physical:
                stats["contact_by"] += 1
        elif token.dep_ in OBJ or token.dep_ in PASSIVE:
            stats["patient"] += 1
            stats["as_patient"][lemma] += 1

# victim framing: contact language in the headline and she isn't the one delivering it
contact_words = framing.rx("victim", "word")
by_her = framing.by_her_rx(surname)

victim = Counter()
victim_examples = defaultdict(list)
for headline in texts:
    if not contact_words.search(headline):
        continue
    for name in players:
        if not name_rx[name].search(headline):
            continue
        if by_her[name].search(headline):
            continue
        victim[name] += 1
        if len(victim_examples[name]) < 4:
            victim_examples[name].append(headline)


## Review sheet — every headline

`data/framing_review.csv` is where we can manually annotate and edit.


In [ ]:
REVIEW = DATA / "framing_review.csv"
BASELINE = DATA / "framing_machine.csv"
MEASURES = ["political", "appearance", "celebrity", "victim"]

def names_of(hits):
    out = defaultdict(set)
    for player, rs in hits.items():
        for r in rs:
            out[r["t"]].add(player)
    return out

appearance_of = names_of(appearance_hits)
celebrity_of = names_of(celebrity_hits)
victim_of = defaultdict(set)
for h in texts:
    if contact_words.search(h):
        for name in players:
            if name_rx[name].search(h) and not by_her[name].search(h):
                victim_of[h].add(name)

machine = {}
for row in headlines:
    t = row["t"]
    machine[t] = {
        "political": hl_political.get(t, ""),
        "appearance": "; ".join(sorted(appearance_of.get(t, []))),
        # an outfit story is celebrity treatment too: the machine prefills
        # the union, and the sheet is what the chart counts
        "celebrity": "; ".join(sorted(celebrity_of.get(t, set()) | appearance_of.get(t, set()))),
        "victim": "; ".join(sorted(victim_of.get(t, []))),
    }

previous_machine = {}
if BASELINE.exists():
    for r in csv.DictReader(open(BASELINE)):
        previous_machine[r["headline"]] = r
hers = {}
if REVIEW.exists():
    rdr = list(csv.DictReader(open(REVIEW)))
    if rdr and all(m in rdr[0] for m in MEASURES):
        hers = {r["headline"]: r for r in rdr}
def her_row(t):
    # Google Sheets eats a leading apostrophe on import, so try both spellings
    return hers.get(t) or hers.get(t.lstrip("'"))

final, edits = {}, 0
for row in headlines:
    t = row["t"]
    final[t] = dict(machine[t])
    h = her_row(t)
    if h and t in previous_machine:
        for m in MEASURES:
            if h.get(m, "") != previous_machine[t].get(m, ""):
                final[t][m] = h.get(m, "")
                edits += 1

# appearance counts as celebrity no matter who marked it — machine or her.
# (To pull a headline out of celebrity entirely, clear its appearance cell too.)
for t in final:
    both = {p.strip() for m in ("celebrity", "appearance")
            for p in final[t][m].split(";") if p.strip()}
    final[t]["celebrity"] = "; ".join(sorted(both))

with REVIEW.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["date", "headline", "players"] + MEASURES)
    for row in headlines:
        t = row["t"]
        named = "; ".join(sorted(p for p, rx in name_rx.items()
                                 if groups[p] == "players" and rx.search(t)))
        w.writerow([row["d"], t, named] + [final[t][m] for m in MEASURES])
with BASELINE.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["headline"] + MEASURES)
    for row in headlines:
        w.writerow([row["t"]] + [machine[row["t"]][m] for m in MEASURES])

# fold the rulings back into the structures everything below uses
by_headline = {r["t"]: r for r in headlines}
def rebuild_hits(measure):
    hits = defaultdict(list)
    for row in headlines:
        for player in [p.strip() for p in final[row["t"]][measure].split(";") if p.strip()]:
            hits[player].append(row)
    return hits
appearance_hits = rebuild_hits("appearance")
celebrity_hits = rebuild_hits("celebrity")
victim = Counter()
victim_examples = defaultdict(list)
for row in headlines:
    for player in [p.strip() for p in final[row["t"]]["victim"].split(";") if p.strip()]:
        victim[player] += 1
        if len(victim_examples[player]) < 4:
            victim_examples[player].append(row["t"])

print(f"review sheet: {len(headlines):,} headlines · {edits} hand edits honored")

Politicization per person


In [ ]:
political = Counter()
group_hits = defaultdict(Counter)
word_hits = defaultdict(Counter)
examples = defaultdict(list)
for row in headlines:
    t = row["t"]
    verdict = final[t]["political"]
    edited = verdict != machine[t]["political"]
    for person, rx in name_rx.items():
        if not rx.search(t):
            continue
        counts = bool(verdict) if edited else (t, person) in pol_ok
        if not counts:
            continue
        political[person] += 1
        own = own_culture_war(person)
        for g, w in hl_fired.get(t, []):
            if g == "culture_war" and any(r.search(w) for r in own):
                continue
            group_hits[person][g] += 1
            word_hits[person][w] += 1
        if len(examples[person]) < 3:
            examples[person].append(t)

people = sorted(
    ({"name": p, "group": groups[p], "n": totals[p], "political": political[p],
      "share": round(political[p] / totals[p], 4),
      "top_groups": "; ".join(g for g, _ in group_hits[p].most_common(3)),
      "matched_words": "; ".join(w for w, _ in word_hits[p].most_common()),
      "examples": examples[p]}
     for p in totals if totals[p] >= MIN_HEADLINES),
    key=lambda r: -r["share"])
rows = [{**{k: v for k, v in p.items() if k != "examples"}, "example": ex}
        for p in people for ex in (p["examples"] or [""])]
pd.options.display.max_colwidth = 1000
pd.DataFrame(people).drop(columns="examples").head(10)

#### Overwrites `src/data/politicization_people.csv`

In [ ]:
framing_parts.append(pd.DataFrame(rows).assign(metric="politicization"))
pd.DataFrame(people).drop(columns="examples").head(10)

In [ ]:
def people_table(hits, example_rows):
    people = sorted(
        ({"name": p, "n": totals[p], "hits": len(hits.get(p, [])),
          "share": round(len(hits.get(p, [])) / totals[p], 4),
          "examples": example_rows(p)}
         for p in totals if totals[p] >= MIN_HEADLINES and groups[p] == "players"),
        key=lambda r: -r["share"])
    return people

def long_rows(people):
    return [{**{k: v for k, v in p.items() if k != "examples"},
             "example_date": ex.get("d", ""), "example_title": ex.get("t", "")}
            for p in people for ex in (p["examples"] or [{}])]

appearance_people = people_table(appearance_hits, lambda p: appearance_hits.get(p, [])[:4])

#### Overwrites `src/data/appearance_talk.csv`

In [ ]:
framing_parts.append(pd.DataFrame(long_rows(appearance_people)).assign(metric="appearance"))

In [ ]:
def cel_examples(p):
    rs = sorted(celebrity_hits.get(p, []), key=lambda r: r["d"], reverse=True)
    return [{"d": r["d"], "t": r["t"]} for r in rs[:4]]
celebrity_people = people_table(celebrity_hits, cel_examples)
pd.DataFrame(celebrity_people).drop(columns="examples").head(10)

#### Overwrites `src/data/celebrity_talk.csv`

In [ ]:
framing_parts.append(pd.DataFrame(long_rows(celebrity_people)).assign(metric="celebrity"))

pd.DataFrame(celebrity_people).drop(columns="examples").head(10)

In [ ]:
people = sorted(
    ({"name": name, "mentions": s["mentions"], "placed": s["agent"] + s["patient"],
      "coverage": round((s["agent"] + s["patient"]) / s["mentions"], 3),
      "agency": round((s["agent"] - s["patient"]) / (s["agent"] + s["patient"]), 3),
      "victim": victim[name], "victim_rate": round(victim[name] / s["mentions"], 4),
      "contact_by": s["contact_by"],
      "top_as_agent": "; ".join(w for w, _ in s["as_agent"].most_common(5)),
      "top_as_patient": "; ".join(w for w, _ in s["as_patient"].most_common(5)),
      "examples": victim_examples[name]}
     for name, s in placed.items() if s["agent"] + s["patient"] >= MIN_PLACED),
    key=lambda r: r["agency"])

rows = [{**{k: v for k, v in p.items() if k != "examples"}, "example": ex}
        for p in people for ex in (p["examples"] or [""])]

#### Overwrites `src/data/agency.csv`

In [ ]:
framing_parts.append(pd.DataFrame(rows).assign(metric="agency"))
pd.DataFrame(people).drop(columns="examples")

## Performance

How much of a player's coverage is about her actual play: points, rebounds, assists, records. Vocabulary in `data/framing_terms.csv` (register `performance`). Machine-only, no review sheet.


In [ ]:
performance_rx = framing.rx("performance", "word")
# the register's scrub rows ("on the rebound", the dating sense) come out
# before the play words are looked for, as for appearance and celebrity
performance_scrub = framing.scrub_rx("performance")
performance_hits = defaultdict(list)   # player -> [{"d", "t"}]
performance_of = defaultdict(set)      # headline -> players named
for row in headlines:
    t = row["t"]
    if not performance_rx.search(performance_scrub.sub(" ", t) if performance_scrub else t):
        continue
    for name in players:
        if name_rx[name].search(t):
            performance_hits[name].append({"d": row["d"], "t": t})
            performance_of[t].add(name)

performance_people = people_table(
    performance_hits,
    lambda p: sorted(performance_hits.get(p, []), key=lambda r: r["d"], reverse=True)[:4])
framing_parts.append(pd.DataFrame(long_rows(performance_people)).assign(metric="performance"))
print(f"{sum(len(v) for v in performance_hits.values())} performance attributions · {len(performance_of)} headlines")
pd.DataFrame(performance_people).drop(columns="examples").head(12)


#### Overwrites `src/data/labeled_headlines.csv`

In [ ]:
with (OUT / "labeled_headlines.csv").open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["headline"] + MEASURES + ["performance"])
    for row in headlines:
        w.writerow([row["t"]] + [final[row["t"]][m] for m in MEASURES]
                   + ["; ".join(sorted(performance_of.get(row["t"], [])))])

## Spot check

In [ ]:
import random

print("— random headlines, with the political groups and words that fired")
for row in random.sample(headlines, 6):
    fired = [g for g, rx in group_rx.items() if rx.search(row["t"])]
    words = {m.group(0).lower() for g in fired for rx_t in term_rx[g] for m in rx_t.finditer(row["t"])}
    print(f"   {row['t'][:86]}")
    print(f"      → {', '.join(fired) or '(not political)'}{' · ' + ', '.join(sorted(words)) if words else ''}")

print("\n— random appearance attributions (the outfit is grammatically hers)")
pool = [(p, r["t"]) for p, rows in appearance_hits.items() for r in rows]
for p, t in random.sample(pool, min(3, len(pool))):
    print(f"   [{p}] {t[:86]}")

print("\n— random celebrity attributions")
pool = [(p, r["t"]) for p, rows in celebrity_hits.items() for r in rows]
for p, t in random.sample(pool, min(3, len(pool))):
    print(f"   [{p}] {t[:86]}")

print("\n— random victim-framing headlines")
pool = [(p, t) for p, ts in victim_examples.items() for t in ts]
for p, t in random.sample(pool, min(3, len(pool))):
    print(f"   [{p}] {t[:86]}")

In [ ]:
# one framing file: every per-player measure, tagged by metric
framing = pd.concat(framing_parts, ignore_index=True)
framing.to_csv(OUT / "framing.csv", index=False)
print("framing.csv:", len(framing), "rows,", framing.metric.value_counts().to_dict())